# Chapter 19: Calibrating the Threshold (Reference)

## Learning Objectives

- Load gates.yml into a RiskConfig via load_config
- Sweep several candidate thresholds against the five worked-example PRs
- Read a pass/hold table and identify where the current default sits
- Explain why the hard ceiling never moves with the threshold

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

PRA_MODE = 'fixture'


## 1. Load the Config

The next cell loads `gates.yml`. You should see the default weights, threshold=70.0, and hard_ceiling_lines=500.

In [2]:
from labs.lab_19_calibrate import load_config, GATES_CONFIG_PATH

config = load_config(GATES_CONFIG_PATH)
print(f"weights: {config.weights}")
print(f"threshold: {config.threshold}")
print(f"hard_ceiling_lines: {config.hard_ceiling_lines}")

weights: {'lines': 30.0, 'files': 25.0, 'critical_paths': 45.0}
threshold: 70.0
hard_ceiling_lines: 500


## 2. Sweep Thresholds Against the Worked Example

The next cell loads the five worked-example PRs and sweeps five candidate thresholds. You should see the pass count grow from 1/5 at threshold=40 to 3/5 at threshold=70, with the large-migration PR held at every threshold (it's blocked by the hard ceiling, not the score).

In [3]:
from labs.lab_19_calibrate import load_worked_example, sweep_thresholds

prs = load_worked_example()
sweep = sweep_thresholds(prs, [40.0, 55.0, 70.0, 85.0, 100.0], config)
for threshold, rows in sweep.items():
    passing = sum(1 for _, _, passes in rows if passes)
    print(f"threshold={threshold:>5.0f}  ({passing}/{len(rows)} would auto-merge)")
    for title, risk, passes in rows:
        print(f"    risk={risk:>6.1f}  {'MERGE' if passes else 'HOLD':<5}  {title}")

threshold=   40  (1/5 would auto-merge)
    risk=   5.9  MERGE  fix: correct typo in sandbox README
    risk=  66.0  HOLD   feat: add greeting helper to sandbox app
    risk=  67.0  HOLD   ci: tweak gate3 threshold comment
    risk= 172.0  HOLD   refactor: reorganize sandbox module layout
    risk= 510.0  HOLD   migrate: restructure sandbox data layer
threshold=   55  (1/5 would auto-merge)
    risk=   5.9  MERGE  fix: correct typo in sandbox README
    risk=  66.0  HOLD   feat: add greeting helper to sandbox app
    risk=  67.0  HOLD   ci: tweak gate3 threshold comment
    risk= 172.0  HOLD   refactor: reorganize sandbox module layout
    risk= 510.0  HOLD   migrate: restructure sandbox data layer
threshold=   70  (3/5 would auto-merge)
    risk=   5.9  MERGE  fix: correct typo in sandbox README
    risk=  66.0  MERGE  feat: add greeting helper to sandbox app
    risk=  67.0  MERGE  ci: tweak gate3 threshold comment
    risk= 172.0  HOLD   refactor: reorganize sandbox module layout
  

## Takeaways & Next Steps

This notebook's takeaway is the sweep table above -- notice threshold=70 is exactly where the small-feature and workflow-touch PRs cross from HOLD to MERGE.

In [4]:
print("Recalibration is editing gates.yml's threshold value and re-running this sweep.")

Recalibration is editing gates.yml's threshold value and re-running this sweep.


---

📖 **Reading companion:** [Chapter 19: Calibrating the Threshold](../learning_modules/chapter_19_calibrating_threshold.md)
